# Objective 5 - Paraphrase Evasion Test
## Objectives
- measure how much accuracy drops when AI-generated text has been paraphrased, and compare against how the 4 existing commercial detectors

1. Load the model/vectorizer, run predictions on this benchmark dataset.
2. Compare my model's accuracy on paraphrased=yes rows vs paraphrased=no rows.
3. Do the same comparison using the 4 existing detector columns already in the dataset (their verdicts are already recorded, no need to run them).
4. Build one summary table: my model + 4 commercial detectors, accuracy on paraphrased vs non-paraphrased text.
5. Write 2-3 sentences interpreting whether paraphrasing is an effective evasion technique across all tools

**Note we decided NOT to retrain the model with paraphrased data — just test the existing model. This keeps results clean and comparable to the other objectives.**

In [19]:
# initiate the main path for import methods
import os
import sys

sys.path.append(os.path.abspath('..'))

In [ ]:
from src.predict import predict_batch
import pandas as pd

paraphrase_df = pd.read_csv("../data/processed/AI_vs_Human_Data_Parapar.csv")
result = predict_batch(paraphrase_df, text_col="text")

In [5]:
import os
import sys
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import accuracy_score


In [6]:
# Add parent directory to path for imports
sys.path.append(os.path.abspath('..'))

print("="*60)
print("Objective 5: Paraphrase Evasion Test")
print("="*60)

Objective 5: Paraphrase Evasion Test


In [7]:
# ============================================================================
# STEP 1: Load the benchmark dataset
# ============================================================================
print("\n[1] Loading benchmark dataset...")
paraphrase_df = pd.read_csv("../data/processed/AI_vs_Human_Data_Parapar.csv")
print(f"✓ Loaded {len(paraphrase_df)} rows, {len(paraphrase_df.columns)} columns")
print(f"  Columns: {paraphrase_df.columns.tolist()}")


[1] Loading benchmark dataset...
✓ Loaded 350 rows, 9 columns
  Columns: ['Source', 'Model', 'Paraphrased', 'Topic', 'Answer', 'ChatGPT', 'Gemini', 'Claude', 'zeroGPT']


In [8]:
# ============================================================================
# STEP 2: Load the trained model and vectorizer
# ============================================================================
print("\n[2] Loading model and vectorizer...")
try:
    with open('../models/baseline_model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('../models/tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    print("✓ Model and vectorizer loaded successfully")
    
    # Check model classes
    if hasattr(model, 'classes_'):
        print(f"  Model classes: {model.classes_}")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    exit()


[2] Loading model and vectorizer...
✓ Model and vectorizer loaded successfully
  Model classes: ['ai' 'human']


In [9]:
# ============================================================================
# STEP 3: Run predictions on the benchmark dataset
# ============================================================================
print("\n[3] Running predictions on benchmark data...")

# Use 'Answer' as the text column (not 'text')
text_col = 'Answer'
print(f"  Using '{text_col}' as text column")

# Clean the data
X_text = paraphrase_df[text_col].fillna('').astype(str)

# Transform using vectorizer
X = vectorizer.transform(X_text)
print(f"  Transformed data shape: {X.shape}")



[3] Running predictions on benchmark data...
  Using 'Answer' as text column
  Transformed data shape: (350, 5000)


In [10]:
# Get predictions
predictions = model.predict(X)

# Map predictions to binary (based on model classes)
if hasattr(model, 'classes_'):
    # If model returns 'ai'/'human', map to 1/0
    if model.classes_[0] in ['ai', 'human']:
        pred_binary = np.array([1 if p == 'ai' else 0 for p in predictions])
    else:
        pred_binary = np.array(predictions, dtype=int)
else:
    pred_binary = np.array(predictions, dtype=int)


In [11]:
# Add predictions to dataframe
paraphrase_df['my_model_pred'] = pred_binary
paraphrase_df['my_model_pred_label'] = ['AI' if p == 1 else 'Human' for p in pred_binary]

print(f"✓ Predictions completed")
print(f"  Distribution: Human={sum(pred_binary==0)}, AI={sum(pred_binary==1)}")


✓ Predictions completed
  Distribution: Human=46, AI=304


In [12]:
# ============================================================================
# STEP 4: Prepare data for analysis
# ============================================================================
print("\n[4] Preparing data for analysis...")

# Convert author to binary (0=Human, 1=AI)
paraphrase_df['true_label'] = paraphrase_df['Source'].apply(
    lambda x: 1 if str(x).strip().upper() == 'AI' else 0
)

# Convert paraphrased to boolean
paraphrase_df['is_paraphrased'] = paraphrase_df['Paraphrased'].apply(
    lambda x: True if str(x).strip().upper() == 'YES' else False
)

print(f"  True labels: Human={sum(paraphrase_df['true_label']==0)}, AI={sum(paraphrase_df['true_label']==1)}")
print(f"  Paraphrased: Yes={sum(paraphrase_df['is_paraphrased']==True)}, No={sum(paraphrase_df['is_paraphrased']==False)}")


[4] Preparing data for analysis...
  True labels: Human=50, AI=300
  Paraphrased: Yes=150, No=200


In [13]:
# ============================================================================
# STEP 5: Compare my model's accuracy on paraphrased vs non-paraphrased
# ============================================================================
print("\n[5] My Model - Accuracy on Paraphrased vs Non-Paraphrased:")

my_results = {}
for group in [True, False]:
    group_data = paraphrase_df[paraphrase_df['is_paraphrased'] == group]
    if len(group_data) > 0:
        acc = accuracy_score(group_data['true_label'], group_data['my_model_pred'])
        my_results[group] = {'accuracy': acc, 'count': len(group_data)}
        label = "Paraphrased" if group else "Non-paraphrased"
        print(f"  {label}: {acc:.2%} ({len(group_data)} samples)")

# Calculate accuracy drop
if True in my_results and False in my_results:
    drop = my_results[False]['accuracy'] - my_results[True]['accuracy']
    print(f"  Accuracy Drop: {drop:.2%}")




[5] My Model - Accuracy on Paraphrased vs Non-Paraphrased:
  Paraphrased: 94.00% (150 samples)
  Non-paraphrased: 83.50% (200 samples)
  Accuracy Drop: -10.50%


In [14]:
# ============================================================================
# STEP 6: Compare commercial detectors (already in dataset)
# ============================================================================
print("\n[6] Commercial Detectors - Accuracy on Paraphrased vs Non-Paraphrased:")

# Define detector columns
detector_cols = ['ChatGPT', 'Gemini', 'Claude', 'zeroGPT']
detector_results = {}

for detector in detector_cols:
    if detector in paraphrase_df.columns:
        print(f"\n  {detector}:")
        
        # Clean and convert detector to binary
        detector_clean = paraphrase_df[detector].fillna('Human').astype(str).str.strip().str.upper()
        paraphrase_df[f'{detector}_binary'] = detector_clean.apply(
            lambda x: 1 if x == 'AI' else 0
        )
        
        detector_results[detector] = {}
        for group in [True, False]:
            group_data = paraphrase_df[paraphrase_df['is_paraphrased'] == group]
            if len(group_data) > 0:
                acc = accuracy_score(group_data['true_label'], group_data[f'{detector}_binary'])
                detector_results[detector][group] = {'accuracy': acc, 'count': len(group_data)}
                label = "Paraphrased" if group else "Non-paraphrased"
                print(f"    {label}: {acc:.2%} ({len(group_data)} samples)")
        
        # Calculate accuracy drop
        if True in detector_results[detector] and False in detector_results[detector]:
            drop = detector_results[detector][False]['accuracy'] - detector_results[detector][True]['accuracy']
            print(f"    Accuracy Drop: {drop:.2%}")



[6] Commercial Detectors - Accuracy on Paraphrased vs Non-Paraphrased:

  ChatGPT:
    Paraphrased: 13.33% (150 samples)
    Non-paraphrased: 69.00% (200 samples)
    Accuracy Drop: 55.67%

  Gemini:
    Paraphrased: 30.67% (150 samples)
    Non-paraphrased: 48.00% (200 samples)
    Accuracy Drop: 17.33%

  Claude:
    Paraphrased: 26.67% (150 samples)
    Non-paraphrased: 35.00% (200 samples)
    Accuracy Drop: 8.33%

  zeroGPT:
    Paraphrased: 38.67% (150 samples)
    Non-paraphrased: 58.00% (200 samples)
    Accuracy Drop: 19.33%


In [15]:

# ============================================================================
# STEP 7: Build summary table
# ============================================================================
print("\n[7] Building summary table...")

summary_data = {
    'Detector': [],
    'Paraphrased Accuracy': [],
    'Non-Paraphrased Accuracy': [],
    'Accuracy Drop': [],
    'Paraphrased Count': [],
    'Non-Paraphrased Count': []
}

# Add my model
if my_results:
    summary_data['Detector'].append('My Model')
    summary_data['Paraphrased Accuracy'].append(my_results.get(True, {}).get('accuracy', np.nan))
    summary_data['Non-Paraphrased Accuracy'].append(my_results.get(False, {}).get('accuracy', np.nan))
    summary_data['Paraphrased Count'].append(my_results.get(True, {}).get('count', 0))
    summary_data['Non-Paraphrased Count'].append(my_results.get(False, {}).get('count', 0))

# Add commercial detectors
for detector, results in detector_results.items():
    summary_data['Detector'].append(detector)
    summary_data['Paraphrased Accuracy'].append(results.get(True, {}).get('accuracy', np.nan))
    summary_data['Non-Paraphrased Accuracy'].append(results.get(False, {}).get('accuracy', np.nan))
    summary_data['Paraphrased Count'].append(results.get(True, {}).get('count', 0))
    summary_data['Non-Paraphrased Count'].append(results.get(False, {}).get('count', 0))

# Calculate accuracy drops
summary_data['Accuracy Drop'] = [
    (non_para - para) if not (np.isnan(para) or np.isnan(non_para)) else np.nan
    for para, non_para in zip(
        summary_data['Paraphrased Accuracy'],
        summary_data['Non-Paraphrased Accuracy']
    )
]

summary_df = pd.DataFrame(summary_data)

# Format percentages
for col in ['Paraphrased Accuracy', 'Non-Paraphrased Accuracy', 'Accuracy Drop']:
    summary_df[col] = summary_df[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )


[7] Building summary table...


In [16]:
# ============================================================================
# STEP 8: Display summary table
# ============================================================================
print("\n" + "="*60)
print("SUMMARY TABLE: Accuracy on Paraphrased vs Non-Paraphrased Text")
print("="*60)
print(summary_df.to_string(index=False))

# ============================================================================
# STEP 9: Calculate overall accuracy
# ============================================================================
print("\n" + "="*60)
print("OVERALL ACCURACY (All Data)")
print("="*60)

# My model
if 'my_model_pred' in paraphrase_df.columns:
    overall_acc = accuracy_score(paraphrase_df['true_label'], paraphrase_df['my_model_pred'])
    print(f"  My Model: {overall_acc:.2%}")

# Commercial detectors
for detector in detector_cols:
    if f'{detector}_binary' in paraphrase_df.columns:
        overall_acc = accuracy_score(paraphrase_df['true_label'], paraphrase_df[f'{detector}_binary'])
        print(f"  {detector}: {overall_acc:.2%}")



SUMMARY TABLE: Accuracy on Paraphrased vs Non-Paraphrased Text
Detector Paraphrased Accuracy Non-Paraphrased Accuracy Accuracy Drop  Paraphrased Count  Non-Paraphrased Count
My Model               94.00%                   83.50%       -10.50%                150                    200
 ChatGPT               13.33%                   69.00%        55.67%                150                    200
  Gemini               30.67%                   48.00%        17.33%                150                    200
  Claude               26.67%                   35.00%         8.33%                150                    200
 zeroGPT               38.67%                   58.00%        19.33%                150                    200

OVERALL ACCURACY (All Data)
  My Model: 88.00%
  ChatGPT: 45.14%
  Gemini: 40.57%
  Claude: 31.43%
  zeroGPT: 49.71%


In [17]:
# ============================================================================
# STEP 10: Interpretation (2-3 sentences)
# ============================================================================
print("\n" + "="*60)
print("INTERPRETATION: Is Paraphrasing an Effective Evasion Technique?")
print("="*60)

# Calculate average drop
drop_values = []
for _, row in summary_df.iterrows():
    if row['Accuracy Drop'] != "N/A":
        drop = float(row['Accuracy Drop'].replace('%', ''))
        drop_values.append(drop)

if drop_values:
    avg_drop = np.mean(drop_values)
    
    print("\n📊 KEY FINDINGS:")
    print(f"  • Average accuracy drop across all detectors: {avg_drop:.1f}%")
    print(f"  • Paraphrasing reduces detection accuracy by {avg_drop:.1f}% on average")
    
    # Find best and worst performers
    para_accs = []
    for idx, row in summary_df.iterrows():
        if row['Paraphrased Accuracy'] != "N/A":
            para_accs.append((row['Detector'], float(row['Paraphrased Accuracy'].replace('%', ''))))
    
    if para_accs:
        best = max(para_accs, key=lambda x: x[1])
        worst = min(para_accs, key=lambda x: x[1])
        print(f"  • Best on paraphrased text: {best[0]} ({best[1]:.1f}%)")
        print(f"  • Worst on paraphrased text: {worst[0]} ({worst[1]:.1f}%)")
    
    # Final interpretation (2-3 sentences)
    print("\n📝 CONCLUSION:")
    if avg_drop > 10:
        print("  Paraphrasing is a HIGHLY EFFECTIVE evasion technique, reducing detection")
        print(f"  accuracy by {avg_drop:.1f}% across all tools. This suggests that current")
        print("  AI text detectors are vulnerable to simple text rewriting, highlighting")
        print("  the need for more robust detection methods.")
    else:
        print("  Paraphrasing has a MODEST impact on detection accuracy, with an average")
        print(f"  drop of only {avg_drop:.1f}%. This suggests current detectors are")
        print("  relatively robust to paraphrasing, though some models like ChatGPT")
        print("  show significant vulnerability.")



INTERPRETATION: Is Paraphrasing an Effective Evasion Technique?

📊 KEY FINDINGS:
  • Average accuracy drop across all detectors: 18.0%
  • Paraphrasing reduces detection accuracy by 18.0% on average
  • Best on paraphrased text: My Model (94.0%)
  • Worst on paraphrased text: ChatGPT (13.3%)

📝 CONCLUSION:
  Paraphrasing is a HIGHLY EFFECTIVE evasion technique, reducing detection
  accuracy by 18.0% across all tools. This suggests that current
  AI text detectors are vulnerable to simple text rewriting, highlighting
  the need for more robust detection methods.


In [18]:
# ============================================================================
# STEP 11: Save results
# ============================================================================
print("\n[8] Saving results...")
os.makedirs('../results', exist_ok=True)
summary_df.to_csv('../results/paraphrase_analysis_summary.csv', index=False)
print("✓ Summary saved to ../results/paraphrase_analysis_summary.csv")

print("\n" + "="*60)
print("✅ Objective 5 - Paraphrase Evasion Test Complete!")
print("="*60)


[8] Saving results...
✓ Summary saved to ../results/paraphrase_analysis_summary.csv

✅ Objective 5 - Paraphrase Evasion Test Complete!
